In [1]:
import json
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import config
from utils.data import load_dataset
from utils.gridding import df_to_gridded_da
from utils.tuning import load_seeded_predictions

sns.set_style("whitegrid")


C:\Users\yvjennig\Anaconda3\envs\time_imputation\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
C:\Users\yvjennig\Anaconda3\envs\time_imputation\Lib\site-packages\timm\optim\optim_factory.py:7: FutureWarning: Importing from timm.optim.optim_factory is deprecated, please import via timm.optim
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.optim", FutureWarning)


# Presentation imputation animation

Animates the spatio-temporal evolution of one parameter (default: potential temperature) across
models and depth levels, for presentation slides.

In [2]:
all_model_names = ["mean", "knn", "missforest", "mlp", "remasker", "mastnet"]
results_dir = Path("F:/PhD/phd_repos/time_imputation/output/testing/")

# Output directory
out_dir = "output/plots/test_evaluation/"
os.makedirs(out_dir, exist_ok=True)
anim_depths = [0, 300, 1500]


In [3]:
# Load RMSE (single-run)
rmse_by_model = {}
for model_name in all_model_names:
    with open(results_dir / model_name / f"model{model_name}_splitfinal_trial0.json") as f:
        meta = json.load(f)
    tm = meta.get("test_metrics") or meta.get("metrics_all") or {}
    rmse_by_model[model_name] = tm.get("Global", {}).get("RMSE", np.nan)

# Load seeded RMSE if available
seed_summary_path = results_dir / "seed_variance" / "seed_variance" / "seed_variance_summary.csv"
if seed_summary_path.exists():
    df_seedvar = pd.read_csv(seed_summary_path)
    rmse_sv = df_seedvar[(df_seedvar["variable"] == "Global") & (df_seedvar["metric"] == "RMSE")]
    for _, row in rmse_sv.iterrows():
        if row["model"] in rmse_by_model:
            rmse_by_model[row["model"]] = row["mean"]

# Model order: worst-to-best global RMSE, MaSTNeT last; seed-mean RMSE where available
model_order = sorted(rmse_by_model, key=lambda m: rmse_by_model[m], reverse=True)
if "mastnet" in model_order:
    model_order.remove("mastnet")
    model_order.append("mastnet")
print(model_order)


['mlp', 'mean', 'knn', 'missforest', 'remasker', 'mastnet']


In [4]:
# Load predictions (seed-averaged where available), inverse-scale to physical units
df = load_dataset()
dfs_by_model = {}

for model_name in model_order:
    # Load predictions
    model_dir = results_dir / model_name
    y_pred_raw, y_true_raw, n_seeds = load_seeded_predictions(model_name, results_dir=results_dir)
    print(f"{model_name}: averaged over {n_seeds} seed(s)")

    # True values where observed, predicted elsewhere
    df_imp = y_true_raw.fillna(y_pred_raw)

    # Inverse-scale to physical units
    scalers = joblib.load(model_dir / "scalers.joblib")
    for k, v in scalers.items():
        if k in df_imp.columns:
            df_imp[k] = v.inverse_transform(df_imp[k].to_numpy().reshape(-1, 1))

    # Attach coordinates
    df_imp[config.coordinates] = df[config.coordinates].values

    dfs_by_model[model_name] = df_imp[df_imp["LEV_M"].isin(anim_depths)].copy()


mlp: averaged over 5 seed(s)
mean: averaged over 1 seed(s)
knn: averaged over 1 seed(s)
missforest: averaged over 5 seed(s)
remasker: averaged over 5 seed(s)
mastnet: averaged over 5 seed(s)


In [5]:
CM = 1 / 2.54  # centimeters -> inches, for slide-sized figures

def generate_presentation_animation(
        dfs_by_model,
        model_order,
        depths=(0, 300, 1500),
        parameter="P_TEMPERATURE",
        cmap="inferno",
        norm=None,
        figsize=(23.4 * CM, 10.7 * CM),
        interval=400,
        save_as=None,
        dpi=300,
        bitrate=4000,
        colorbar_label=None,
):
    """
    Animate the spatio-temporal evolution of `parameter` across models and depths.

    Uses an equal-aspect PlateCarree projection (not "auto") to avoid distorting lat/lon.

    Parameters
    ----------
    dfs_by_model : dict[str, pd.DataFrame]
        Per-model DataFrame (physical units, not time-averaged) with columns
        DATEANDTIME, LATITUDE, LONGITUDE, LEV_M and `parameter`. Only rows at
        `depths` are needed.
    model_order : list of str
        Column order left to right.
    depths : sequence of float
        Depth levels [m] to show, one per row, top to bottom.
    figsize : tuple
        Figure size in inches. Defaults to a 23.4 x 10.7 cm slide panel.
    save_as : str, optional
        Path ending in .mp4 or .gif. If omitted, the animation is shown instead.
    dpi : int
        Resolution of the saved animation.
    bitrate : int
        Bitrate (kbps) for the mp4 writer, ignored for .gif.
    colorbar_label : str, optional
        Overrides the default label.

    Returns
    -------
    matplotlib.animation.FuncAnimation
    """
    # Map to xarray grid
    grids = {m: df_to_gridded_da(dfs_by_model[m], value_col=parameter).sel(depth=list(depths))
             for m in model_order}

    # Ensure time steps match across models
    times = grids[model_order[0]].time.values
    for m in model_order[1:]:
        if not np.array_equal(grids[m].time.values, times):
            raise ValueError(f"Time steps for model '{m}' do not match '{model_order[0]}'")

    # Shared colour bar across all spatio-temporal locations and models
    vmin = min(float(np.nanmin(grids[m].values)) for m in model_order)
    vmax = max(float(np.nanmax(grids[m].values)) for m in model_order)

    # One row per depth level, one column per model
    nrows, ncols = len(depths), len(model_order)

    # Figure setup
    fig, axs = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=figsize,
        subplot_kw={"projection": ccrs.PlateCarree()},
        layout="compressed",
    )
    axs = np.atleast_2d(axs)  # keep 2D indexing even when nrows or ncols == 1
    fig.get_layout_engine().set(w_pad=0.02, h_pad=0.02, wspace=0.02, hspace=0.01)

    # Generate all colour meshes
    pcm_list = []
    pcm_grid = {}
    for di, depth in enumerate(depths):
        for mi, m in enumerate(model_order):
            ax = axs[di, mi]
            da_full = grids[m]

            # Map settings
            ax.set_xlim(float(da_full.lon.min()), float(da_full.lon.max()))
            ax.set_ylim(float(da_full.lat.min()), float(da_full.lat.max()))
            ax.set_aspect("equal")
            ax.add_feature(cfeature.LAND, facecolor="lightgrey", zorder=0)
            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)

            # Draw the first frame (time=0) to initialise the mesh (later frames update the array)
            da2d = da_full.isel(time=0).sel(depth=depth)
            if norm is not None:
                pcm = ax.pcolormesh(da2d.lon, da2d.lat, da2d,
                                    cmap=cmap, norm=norm, shading="nearest",
                                    transform=ccrs.PlateCarree())
            else:
                pcm = ax.pcolormesh(da2d.lon, da2d.lat, da2d,
                                    cmap=cmap, vmin=vmin, vmax=vmax, shading="nearest",
                                    transform=ccrs.PlateCarree())
            pcm_list.append(pcm)
            pcm_grid[(di, mi)] = pcm

            # Grid line formatting
            gl = ax.gridlines(draw_labels=True, linewidth=0.3, linestyle="--", alpha=0.5)
            gl.top_labels = False
            gl.right_labels = False
            gl.xformatter = LONGITUDE_FORMATTER
            gl.yformatter = LATITUDE_FORMATTER
            gl.xlabel_style = {"size": 8}
            gl.ylabel_style = {"size": 8}
            gl.left_labels = (mi == 0)
            gl.bottom_labels = (di == nrows - 1)

            # Add ylabel for first model column
            if mi == 0:
                ax.set_ylabel(f"{int(depth)} m", labelpad=25)
                ax.set_yticks([])
                ax.yaxis.set_visible(True)

            # Add title for first depth row
            if di == 0:
                ax.set_title(config.model_name_map.get(m, m))

    # Shared colour bar
    cbar = fig.colorbar(pcm_list[0], ax=axs, orientation="vertical", shrink=0.9, aspect=30, pad=0.02)
    cbar.set_label(colorbar_label if colorbar_label is not None else config.parameter_name_unit_map.get(parameter, parameter))

    # Animated time indicator
    time_label = fig.suptitle(f"Year: {times[0]}", fontsize=12)

    def update(frame_idx):
        # Update every panel without using the pre-computed meshes
        for di, depth in enumerate(depths):
            for mi, m in enumerate(model_order):
                arr = grids[m].isel(time=frame_idx).sel(depth=depth).values
                pcm_grid[(di, mi)].set_array(arr.ravel())
        time_label.set_text(f"Year: {times[frame_idx]}")
        return pcm_list + [time_label]

    # Animation
    anim = animation.FuncAnimation(fig, update, frames=len(times), interval=interval, blit=False)

    # Save (writer depends on file extension)
    if save_as:
        save_as = str(save_as)
        if save_as.endswith(".mp4"):
            writer = animation.FFMpegWriter(fps=1000 / interval, bitrate=bitrate)
        elif save_as.endswith(".gif"):
            writer = animation.PillowWriter(fps=1000 / interval)
        else:
            raise ValueError("save_as must end with .mp4 or .gif")
        anim.save(save_as, writer=writer, dpi=dpi)
        print(f"Animation saved as {save_as}")
        plt.close(fig)
    else:
        plt.show()

    return anim


In [6]:
generate_presentation_animation(
    dfs_by_model=dfs_by_model,
    model_order=model_order,
    depths=anim_depths,
    parameter="P_TEMPERATURE",
    dpi=600,
    save_as=out_dir + "presentation_imputation_P_TEMPERATURE.mp4",
)


Animation saved as output/plots/test_evaluation/presentation_imputation_P_TEMPERATURE.mp4
